In [0]:
# Databricks notebook source
# DBTITLE 1,Install Dependencies

%pip install --upgrade numpy==1.26.4 sentence-transformers faiss-cpu tqdm requests
dbutils.library.restartPython()

In [0]:
import gc
import json
import logging
import os
import sys
import time

import numpy as np
import pandas as pd
import requests

# ── Secrets ──────────────────────────────────────────────────────────────────
_secrets     = json.loads(dbutils.fs.head("dbfs:/Workspace/Users/mqwebster238@gmail.com/secrets.json"))
TMDB_API_KEY = _secrets["TMDB_API_KEY"]

# ── Custom modules ────────────────────────────────────────────────────────────
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from features  import build_embedding_input, get_embedding_tier
from model_cb  import build_faiss_index, save_index, load_index, query_index

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── Paths (all controlled here, never hardcoded below) ────────────────────────
OUTPUTS_DIR       = "/Volumes/movie_recsys/data/outputs"
META_CLEAN_PATH   = f"{OUTPUTS_DIR}/meta_clean.parquet"       # key: parent_asin  (748,224 rows)
REVIEWS_PATH      = f"{OUTPUTS_DIR}/reviews_5core.parquet"    # key: parent_asin  (7.5M rows)
MOST_HELPFUL_PATH = f"{OUTPUTS_DIR}/most_helpful.parquet"     # checkpoint
TMDB_CHECKPOINT   = f"{OUTPUTS_DIR}/tmdb_enriched.parquet"    # checkpoint
EMBEDDINGS_PATH   = f"{OUTPUTS_DIR}/embeddings.npy"           # float32 (n, 384)
ASIN_INDEX_PATH   = f"{OUTPUTS_DIR}/asin_index.npy"           # parent_asin aligned to embeddings
FAISS_INDEX_PATH  = f"{OUTPUTS_DIR}/faiss_index.bin"          # IVF-Flat index

# ── Tuning params ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"
EMBEDDING_DIM    = 384
BATCH_SIZE       = 256
N_CLUSTERS       = 256
MAX_REVIEW_WORDS = 256
CHECKPOINT_EVERY = 100   # save embeddings.npy every N batches
LOG_EVERY        = 10

# Set FORCE_* = True only if you want to throw away an existing checkpoint
# and regenerate that stage from scratch. Leave False for normal runs.
FORCE_MOST_HELPFUL = False
FORCE_TMDB         = False
FORCE_EMBEDDINGS   = False

TMDB_SEARCH_URL  = "https://api.themoviedb.org/3/search/movie"
TMDB_SLEEP       = 1.0 / 40          # 40 req/s free tier
SPOT_CHECK_TITLES = ["The Dark Knight", "Toy Story", "The Godfather"]
SPOT_CHECK_K      = 5

In [0]:

# ── 1a. meta_clean ────────────────────────────────────────────────────────────
log.info("Loading meta_clean from %s", META_CLEAN_PATH)
meta = pd.read_parquet(META_CLEAN_PATH)

# Hard contract: this file MUST have parent_asin (validated by S02 notebook)
assert "parent_asin" in meta.columns, (
    f"meta_clean.parquet is missing 'parent_asin'. "
    f"Columns present: {list(meta.columns)}. "
    f"Re-run the EDA notebook and check S02_validate_eda_outputs."
)
log.info("meta_clean: %d rows", len(meta))

# ── 1b. most_helpful ─────────────────────────────────────────────────────────
if not FORCE_MOST_HELPFUL and os.path.exists(MOST_HELPFUL_PATH):
    log.info("most_helpful checkpoint found — loading.")
    most_helpful = pd.read_parquet(MOST_HELPFUL_PATH)
else:
    log.info("Building most_helpful from reviews (this takes a minute)…")
    reviews = pd.read_parquet(REVIEWS_PATH, columns=["parent_asin", "helpful_vote", "text"])
    most_helpful = (
        reviews
        .sort_values("helpful_vote", ascending=False)
        .groupby("parent_asin", as_index=False)
        .first()[["parent_asin", "text"]]
        .rename(columns={"text": "most_helpful_review"})
    )
    del reviews
    gc.collect()
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    most_helpful.to_parquet(MOST_HELPFUL_PATH, index=False)
    log.info("most_helpful saved → %s", MOST_HELPFUL_PATH)

assert "parent_asin" in most_helpful.columns, "most_helpful is missing 'parent_asin'"
log.info("most_helpful: %d rows", len(most_helpful))

# ── 1c. Merge ─────────────────────────────────────────────────────────────────
meta = meta.merge(most_helpful, on="parent_asin", how="left")
log.info("After merge: %d rows (should still be 748,224)", len(meta))
del most_helpful
gc.collect()

In [0]:

def _is_present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, float) and pd.isna(v):
        return False
    s = str(v).strip()
    return bool(s) and s.lower() != "nan"

def _clean(v):
    return str(v).strip() if _is_present(v) else None

def _coalesce(primary, fallback):
    return _clean(primary) if _is_present(primary) else _clean(fallback)

# Clean all string columns in-place
_STR_COLS = [
    "title", "genres_str", "description_str", "most_helpful_review",
    "tmdb_title", "tmdb_description", "tmdb_genres",
    "title_final", "genres_final", "description_final",
]
for col in _STR_COLS:
    if col in meta.columns:
        meta[col] = meta[col].apply(_clean)

log.info("String columns cleaned.")

In [0]:

def _fetch_tmdb(title: str, api_key: str, session: requests.Session):
    try:
        resp = session.get(
            TMDB_SEARCH_URL,
            params={"api_key": api_key, "query": title, "language": "en-US", "page": 1},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        top = results[0]
        return {
            "title":       top.get("title", ""),
            "description": top.get("overview", ""),
            "genres":      "|".join(str(g) for g in top.get("genre_ids", [])),
        }
    except Exception as exc:
        log.warning("TMDB fetch failed for '%s': %s", title, exc)
        return None

# Identify Tier 4 before checking checkpoint (needed either way)
meta["_pre_emb"] = meta.apply(
    lambda r: build_embedding_input(
        r["title"], r["genres_str"], r["description_str"], r.get("most_helpful_review"),
        max_review_words=MAX_REVIEW_WORDS,
    ), axis=1
)
tier4_mask = meta["_pre_emb"].isna()
tier4_df   = meta.loc[tier4_mask, ["parent_asin", "title"]].copy()
log.info("Tier 4 items needing TMDB: %d / %d (%.1f%%)",
         len(tier4_df), len(meta), 100 * len(tier4_df) / len(meta))

if not FORCE_TMDB and os.path.exists(TMDB_CHECKPOINT):
    log.info("TMDB checkpoint found — loading.")
    tmdb_enriched = pd.read_parquet(TMDB_CHECKPOINT)
else:
    assert TMDB_API_KEY, "TMDB_API_KEY missing from secrets.json"
    records = []
    session = requests.Session()
    total   = len(tier4_df)
    for i, (_, row) in enumerate(tier4_df.iterrows()):
        if i % 500 == 0:
            log.info("TMDB: %d / %d (%.0f%%)", i, total, 100 * i / max(total, 1))
        result = _fetch_tmdb(row["title"] or "", TMDB_API_KEY, session)
        records.append({
            "parent_asin":      row["parent_asin"],    # ← always parent_asin
            "tmdb_title":       result["title"]       if result else None,
            "tmdb_description": result["description"] if result else None,
            "tmdb_genres":      result["genres"]      if result else None,
        })
        time.sleep(TMDB_SLEEP)
    session.close()
    tmdb_enriched = pd.DataFrame(records)
    tmdb_enriched.to_parquet(TMDB_CHECKPOINT, index=False)
    log.info("TMDB checkpoint saved → %s", TMDB_CHECKPOINT)

assert "parent_asin" in tmdb_enriched.columns, "tmdb_enriched is missing 'parent_asin'"

meta = meta.merge(tmdb_enriched, on="parent_asin", how="left")
meta.drop(columns=["_pre_emb"], inplace=True)

# Clean tmdb_* cols after merge
for col in ["tmdb_title", "tmdb_description", "tmdb_genres"]:
    if col in meta.columns:
        meta[col] = meta[col].apply(_clean)

# Coalesce: Amazon first, TMDB fallback
meta["title_final"]       = meta.apply(lambda r: _coalesce(r["title"],           r.get("tmdb_title")),       axis=1)
meta["genres_final"]      = meta.apply(lambda r: _coalesce(r["genres_str"],      r.get("tmdb_genres")),      axis=1)
meta["description_final"] = meta.apply(lambda r: _coalesce(r["description_str"], r.get("tmdb_description")), axis=1)

log.info("Post-TMDB coverage — title: %.1f%%, genres: %.1f%%, desc: %.1f%%",
         meta["title_final"].notna().mean() * 100,
         meta["genres_final"].notna().mean() * 100,
         meta["description_final"].notna().mean() * 100)

In [0]:

meta["embedding_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review"),
        max_review_words=MAX_REVIEW_WORDS,
    ), axis=1
)
meta["embedding_tier"] = meta.apply(
    lambda r: get_embedding_tier(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review"),
    ), axis=1
)

true_gaps  = meta["embedding_input"].isna()
embeddable = meta[~true_gaps].reset_index(drop=True)

log.info("Tier distribution:")
for tier, count in meta["embedding_tier"].value_counts().sort_index().items():
    log.info("  Tier %d: %6d  (%.1f%%)", tier, count, 100 * count / len(meta))
log.info("True gaps (will be skipped): %d", true_gaps.sum())
log.info("Items to embed: %d", len(embeddable))

# The list of parent_asin values aligned to the embedding matrix rows
asins = embeddable["parent_asin"].tolist()
texts = embeddable["embedding_input"].tolist()
n     = len(texts)

In [0]:

from sentence_transformers import SentenceTransformer

if not FORCE_EMBEDDINGS and os.path.exists(EMBEDDINGS_PATH) and os.path.exists(ASIN_INDEX_PATH):
    existing_emb  = np.load(EMBEDDINGS_PATH)
    existing_asin = np.load(ASIN_INDEX_PATH, allow_pickle=True).tolist()
    n_done = len(existing_emb)
    log.info("Checkpoint found: %d / %d items already embedded (%.1f%%)",
             n_done, n, 100 * n_done / n)
    if n_done >= n:
        log.info("Embedding already complete — skipping.")
        all_embeddings = existing_emb
        all_asins      = existing_asin
        total_time     = None
    else:
        start_batch    = n_done // BATCH_SIZE
        all_embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)
        all_embeddings[:n_done] = existing_emb
        all_asins      = asins   # full ordered list; first n_done are already done
        del existing_emb
        gc.collect()
else:
    start_batch    = 0
    all_embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)
    all_asins      = asins
    n_done         = 0

# Only run the loop if there is work left
if n_done < n:
    log.info("Loading model: %s", EMBEDDING_MODEL)
    model      = SentenceTransformer(EMBEDDING_MODEL)
    n_batches  = (n + BATCH_SIZE - 1) // BATCH_SIZE
    t_start    = time.time()

    for batch_idx in range(start_batch, n_batches):
        lo = batch_idx * BATCH_SIZE
        hi = min(lo + BATCH_SIZE, n)

        all_embeddings[lo:hi] = model.encode(
            texts[lo:hi],
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,   # unit-norm → cosine via inner product
        ).astype(np.float32)

        if (batch_idx + 1) % LOG_EVERY == 0 or batch_idx == n_batches - 1:
            elapsed  = time.time() - t_start
            done_now = hi - start_batch * BATCH_SIZE
            rate     = done_now / elapsed if elapsed > 0 else 0
            eta      = (n - hi) / rate if rate > 0 else 0
            log.info("Batch %d/%d | items %d–%d | %.0f items/s | ETA %.0f s",
                     batch_idx + 1, n_batches, lo, hi - 1, rate, eta)

        if (batch_idx + 1) % CHECKPOINT_EVERY == 0:
            os.makedirs(OUTPUTS_DIR, exist_ok=True)
            np.save(EMBEDDINGS_PATH, all_embeddings[:hi])
            np.save(ASIN_INDEX_PATH, np.array(asins[:hi], dtype=object))
            log.info("  ✓ checkpoint saved at item %d", hi)

    total_time = time.time() - t_start
    log.info("Embedding loop done in %.0f s", total_time)

    # Final save of complete arrays
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    np.save(EMBEDDINGS_PATH, all_embeddings)
    np.save(ASIN_INDEX_PATH, np.array(all_asins, dtype=object))
    log.info("Final embeddings saved: %d items", n)

    del model
    gc.collect()

In [0]:

# Reload from disk to guarantee we use exactly what was saved
all_embeddings = np.load(EMBEDDINGS_PATH)
all_asins      = np.load(ASIN_INDEX_PATH, allow_pickle=True).tolist()

assert len(all_embeddings) == len(all_asins), (
    f"Mismatch: {len(all_embeddings)} embeddings vs {len(all_asins)} ASINs"
)
assert all_embeddings.shape[1] == EMBEDDING_DIM, (
    f"Wrong dim: {all_embeddings.shape[1]} (expected {EMBEDDING_DIM})"
)
log.info("Building FAISS index: %d items × %d dims", len(all_embeddings), EMBEDDING_DIM)

n_clusters_actual = min(N_CLUSTERS, len(all_embeddings))
index = build_faiss_index(all_embeddings, n_clusters=n_clusters_actual)
save_index(index, FAISS_INDEX_PATH)

assert index.ntotal == len(all_embeddings), (
    f"FAISS ntotal {index.ntotal} != embeddings {len(all_embeddings)}"
)
log.info("FAISS index saved: ntotal=%d, size=%.1f MB",
         index.ntotal, os.path.getsize(FAISS_INDEX_PATH) / 1024**2)

In [0]:
print("=" * 65)
print("JOB 1 VALIDATION")
print("=" * 65)

results = {}
def check(name, passed, detail=""):
    results[name] = passed
    tag = "✅" if passed else "❌"
    print(f"  {tag}  {name}" + (f"  [{detail}]" if detail else ""))

n_emb  = len(all_embeddings)
pct    = n_emb / len(meta) * 100
norms  = np.linalg.norm(all_embeddings, axis=1)

print("\nT1 · Completeness")
check("Has > 400K embeddings",   n_emb > 400_000,          f"{n_emb:,}")
check("Coverage ≥ 55%",          pct >= 55,                 f"{pct:.1f}%")

print("\nT2 · Shape")
check("Dim = 384",               all_embeddings.shape[1] == 384)
check("ASIN count matches",      len(all_asins) == n_emb)
check("FAISS ntotal matches",    index.ntotal == n_emb)

print("\nT3 · Quality")
check("No NaN",                  not np.isnan(all_embeddings).any())
check("No zero vectors",         (np.abs(all_embeddings).sum(axis=1) == 0).sum() == 0)
check("Norms ≈ 1.0",             0.95 <= norms.mean() <= 1.05,  f"{norms.mean():.3f}")

print("\nT4 · FAISS")
dists, idxs = query_index(index, all_embeddings[0:1], k=6)
check("Returns 6 results",       dists.shape == (1, 6))
check("Self-query dist ≈ 0",     dists[0][0] < 0.001)

print("\nT5 · parent_asin coverage")
meta_asins_set = set(meta["parent_asin"].astype(str))
emb_asins_set  = set(str(a) for a in all_asins)
overlap_pct    = len(emb_asins_set & meta_asins_set) / n_emb
check("No null ASINs",           sum(1 for a in all_asins if not str(a).strip()) == 0)
check("≥ 90% overlap with meta", overlap_pct >= 0.90,  f"{overlap_pct*100:.1f}%")

print("\nT6 · Spot-check nearest neighbours")
asin_to_idx   = {a: i for i, a in enumerate(all_asins)}
asin_to_title = meta.set_index("parent_asin")["title_final"].fillna("(unknown)").to_dict()
for seed_title in SPOT_CHECK_TITLES:
    matches = meta[meta["title_final"].str.contains(seed_title, case=False, na=False)]
    if matches.empty:
        print(f"    '{seed_title}': not in metadata — skipped")
        continue
    seed_asin = matches.iloc[0]["parent_asin"]
    seed_idx  = asin_to_idx.get(seed_asin)
    if seed_idx is None:
        print(f"    '{seed_title}': not in embedded set — skipped")
        continue
    dists2, idxs2 = query_index(index, all_embeddings[seed_idx:seed_idx+1], k=SPOT_CHECK_K+1)
    print(f"\n    Seed: {asin_to_title.get(seed_asin, seed_asin)}")
    for dist, ni in zip(dists2[0], idxs2[0]):
        if ni == seed_idx:
            continue
        print(f"      → {asin_to_title.get(all_asins[int(ni)], '(unknown)')}  [L2={dist:.4f}]")

print("\n" + "=" * 65)
passed = sum(results.values())
failed = len(results) - passed
print(f"RESULT: {passed} passed, {failed} failed")
# NOTE: removed dbutils.notebook.exit() so you can see the full output